# Plot oneway/lift/double-lift from saved models

This notebook loads saved models from their `output_dir/model` and uses them
to generate predicted oneway, lift, and double-lift plots.

For FT -> XGB/ResNet two-step runs:
- Use the Step-2 configs generated by `Train_FT_Embed_XGBResN.ipynb` to locate
  the XGB/ResNet models.
- Load the FT model from Step-1 to generate embeddings on the raw dataset.
- Use raw features for oneway plots while predictions use embedding features.

If BayesOptModel preprocessing fails with "duplicate labels", use the
lightweight predictor + plotting helpers in this notebook (default).

Note: models saved with the new format include preprocess artifacts and
best params inside the model file. If you trained with an older version,
re-save the model to use this workflow.


In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / 'ins_pricing').exists():
    for parent in repo_root.parents:
        if (parent / 'ins_pricing').exists():
            repo_root = parent
            sys.path.insert(0, str(repo_root))
            break

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import json
import pandas as pd
import torch

from ins_pricing.cli.utils.cli_common import split_train_test
from ins_pricing.modelling.plotting import (
    PlotStyle,
    plot_double_lift_curve,
    plot_lift_curve,
    plot_oneway,
)
from ins_pricing.modelling.plotting.common import finalize_figure, plt
from ins_pricing.production.predict import load_predictor_from_config


In [ ]:
# Base config used for data splitting/plotting (typically the Step-2 XGB config).
cfg_path = repo_root / 'ins_pricing/examples/modelling/config_xgb_from_ft_unsupervised.json'

# Step-2 configs to locate each model's output_dir.
xgb_cfg_path = repo_root / 'ins_pricing/examples/modelling/config_xgb_from_ft_unsupervised.json'
resn_cfg_path = repo_root / 'ins_pricing/examples/modelling/config_resn_from_ft_unsupervised_ddp.json'

# Step-1 FT config used to generate embeddings on raw data.
ft_cfg_path = repo_root / 'ins_pricing/examples/modelling/config_ft_unsupervised_ddp_embed.json'

cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
xgb_cfg = json.loads(xgb_cfg_path.read_text(encoding='utf-8'))
resn_cfg = json.loads(resn_cfg_path.read_text(encoding='utf-8'))
ft_cfg = json.loads(ft_cfg_path.read_text(encoding='utf-8'))


def _dedupe_list(values):
    seen = set()
    out = []
    for item in values or []:
        if item in seen:
            continue
        seen.add(item)
        out.append(item)
    return out


def _drop_duplicate_columns(df, label):
    if df.columns.duplicated().any():
        dupes = [str(x) for x in df.columns[df.columns.duplicated()]]
        print(f"[Warn] {label}: dropping duplicate columns: {sorted(set(dupes))}")
        return df.loc[:, ~df.columns.duplicated()].copy()
    return df


model_name = f"{cfg['model_list'][0]}_{cfg['model_categories'][0]}"
raw_data_dir = (ft_cfg_path.parent / ft_cfg['data_dir']).resolve()
raw_path = raw_data_dir / f"{model_name}.csv"
raw = pd.read_csv(raw_path)
raw = _drop_duplicate_columns(raw, "raw")
raw = raw.reset_index(drop=True)
raw.fillna(0, inplace=True)

ft_output_dir = (ft_cfg_path.parent / ft_cfg['output_dir']).resolve()
ft_prefix = ft_cfg.get('ft_feature_prefix', 'ft_emb')
raw_feature_list = _dedupe_list(ft_cfg.get('feature_list') or [])
raw_categorical_features = _dedupe_list(ft_cfg.get('categorical_features') or [])

if ft_cfg.get('geo_feature_nmes'):
    raise ValueError('FT inference with geo tokens is not supported in this example.')

holdout_ratio = cfg.get('holdout_ratio', cfg.get('prop_test', 0.25))
split_strategy = cfg.get('split_strategy', 'random')
split_group_col = cfg.get('split_group_col')
split_time_col = cfg.get('split_time_col')
split_time_ascending = cfg.get('split_time_ascending', True)
rand_seed = cfg.get('rand_seed', 13)

# Always split raw data for plotting features.
train_raw, test_raw = split_train_test(
    raw,
    holdout_ratio=holdout_ratio,
    strategy=split_strategy,
    group_col=split_group_col,
    time_col=split_time_col,
    time_ascending=split_time_ascending,
    rand_seed=rand_seed,
    reset_index_mode='none',
    ratio_label='holdout_ratio',
)
train_raw = _drop_duplicate_columns(train_raw, "train_raw")
test_raw = _drop_duplicate_columns(test_raw, "test_raw")

# Use precomputed embedding data by default to avoid duplicate-column errors.
use_runtime_ft_embedding = False


def _load_torch_payload(path):
    return torch.load(path, map_location='cpu')


if use_runtime_ft_embedding:
    # Load FT model and generate embeddings on raw splits.
    ft_model_path = ft_output_dir / 'model' / f"01_{model_name}_FTTransformer.pth"
    ft_payload = _load_torch_payload(ft_model_path)
    if isinstance(ft_payload, dict) and 'model' in ft_payload:
        ft_model = ft_payload['model']
    else:
        ft_model = ft_payload


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if hasattr(ft_model, 'device'):
        ft_model.device = device
    if hasattr(ft_model, 'to'):
        ft_model.to(device)
    if hasattr(ft_model, 'ft'):
        ft_model.ft.to(device)

    emb_train = ft_model.predict(train_raw, return_embedding=True)
    emb_cols = [f"pred_{ft_prefix}_{i}" for i in range(emb_train.shape[1])]
    train_df = train_raw.copy()
    train_df[emb_cols] = emb_train

    emb_test = ft_model.predict(test_raw, return_embedding=True)
    test_df = test_raw.copy()
    test_df[emb_cols] = emb_test
else:
    # Load embedding-augmented dataset from Step-2 config.
    embed_data_dir = (cfg_path.parent / cfg['data_dir']).resolve()
    embed_path = embed_data_dir / f"{model_name}.csv"
    embed_df = pd.read_csv(embed_path)
    embed_df = _drop_duplicate_columns(embed_df, "embed")
    embed_df = embed_df.reset_index(drop=True)
    embed_df.fillna(0, inplace=True)
    if len(embed_df) != len(raw):
        raise ValueError(
            f"Row count mismatch: raw={len(raw)}, embed={len(embed_df)}. "
            "Cannot align predictions to raw features."
        )

    train_df = embed_df.loc[train_raw.index].copy()
    test_df = embed_df.loc[test_raw.index].copy()

# De-duplicate feature lists for optional fallback usage.
feature_list = _dedupe_list(cfg.get('feature_list') or [])
categorical_features = _dedupe_list(cfg.get('categorical_features') or [])

output_dir = cfg.get('output_dir', './Results')
output_dir = str((cfg_path.parent / output_dir).resolve())
plot_path_style = str(cfg.get('plot_path_style', 'nested') or 'nested').strip().lower()


In [ ]:
def _safe_tag(value: str) -> str:
    return (
        value.strip()
        .replace(' ', '_')
        .replace('/', '_')
        .replace('\\', '_')
        .replace(':', '_')
    )


def _resolve_plot_path(subdir: str, filename: str) -> str:
    plot_root = Path(output_dir) / 'plot'
    if plot_path_style in {'flat', 'root'}:
        return str((plot_root / filename).resolve())
    if subdir:
        return str((plot_root / subdir / filename).resolve())
    return str((plot_root / filename).resolve())


def _load_predictor(cfg_path: Path, model_key: str):
    return load_predictor_from_config(cfg_path, model_key, model_name=model_name)


In [ ]:
model_keys = ['xgb', 'resn']
model_cfg_map = {
    'xgb': xgb_cfg_path,
    'resn': resn_cfg_path,
}

predictors = {}
for key in model_keys:
    cfg_path = model_cfg_map.get(key)
    if cfg_path is None:
        raise ValueError(f"Missing config for model key: {key}")
    predictors[key] = _load_predictor(cfg_path, key)

pred_train = {}
pred_test = {}
for key, predictor in predictors.items():
    pred_train[key] = predictor.predict(train_df).reshape(-1)
    pred_test[key] = predictor.predict(test_df).reshape(-1)
    if len(pred_train[key]) != len(train_df):
        raise ValueError(f"Train prediction length mismatch for {key}")
    if len(pred_test[key]) != len(test_df):
        raise ValueError(f"Test prediction length mismatch for {key}")

# Build plotting frames on raw features.
plot_train = train_raw.copy()
plot_test = test_raw.copy()

for key in model_keys:
    plot_train[f'pred_{key}'] = pred_train[key]
    plot_test[f'pred_{key}'] = pred_test[key]

weight_col = cfg['weight']
target_col = cfg['target']

if weight_col not in plot_train.columns:
    plot_train[weight_col] = 1.0
if weight_col not in plot_test.columns:
    plot_test[weight_col] = 1.0

if target_col in plot_train.columns:
    plot_train['w_act'] = plot_train[target_col] * plot_train[weight_col]
if target_col in plot_test.columns:
    plot_test['w_act'] = plot_test[target_col] * plot_test[weight_col]

if 'w_act' not in plot_train.columns or plot_train['w_act'].isna().all():
    print('[Plot] Missing target values in train split; skip plots.')
else:
    n_bins = cfg.get('plot', {}).get('n_bins', 10)

    oneway_features = raw_feature_list or feature_list
    oneway_categorical = set(raw_categorical_features or categorical_features)

    for pred_key, pred_label in [('xgb', 'Xgboost'), ('resn', 'ResNet')]:
        pred_col = f'pred_{pred_key}'
        if pred_col not in plot_train.columns:
            continue
        pred_tag = _safe_tag(pred_label or pred_col)
        for feature in oneway_features:
            if feature not in plot_train.columns:
                continue
            save_path = _resolve_plot_path(
                f"{model_name}/oneway/post",
                f"00_{model_name}_{feature}_oneway_{pred_tag}.png",
            )
            plot_oneway(
                plot_train,
                feature=feature,
                weight_col=weight_col,
                target_col='w_act',
                pred_col=pred_col,
                pred_label=pred_label,
                n_bins=n_bins,
                is_categorical=feature in oneway_categorical,
                save_path=save_path,
                show=False,
            )

    datasets = []
    if 'w_act' in plot_train.columns and not plot_train['w_act'].isna().all():
        datasets.append(('Train Data', plot_train))
    if 'w_act' in plot_test.columns and not plot_test['w_act'].isna().all():
        datasets.append(('Test Data', plot_test))

    def _plot_lift_for_model(pred_key: str, pred_label: str) -> None:
        if not datasets:
            return
        style = PlotStyle()
        fig, axes = plt.subplots(1, len(datasets), figsize=(11, 5))
        if len(datasets) == 1:
            axes = [axes]
        for ax, (title, data) in zip(axes, datasets):
            pred_col = f'pred_{pred_key}'
            if pred_col not in data.columns:
                continue
            plot_lift_curve(
                data[pred_col].values,
                data['w_act'].values,
                data[weight_col].values,
                n_bins=n_bins,
                title=f"Lift Chart on {title}",
                pred_label='Predicted',
                act_label='Actual',
                weight_label='Earned Exposure',
                pred_weighted=False,
                actual_weighted=True,
                ax=ax,
                show=False,
                style=style,
            )
        plt.subplots_adjust(wspace=0.3)
        filename = f"01_{model_name}_{_safe_tag(pred_label)}_lift.png"
        save_path = _resolve_plot_path(f"{model_name}/lift", filename)
        finalize_figure(fig, save_path=save_path, show=False, style=style)

    _plot_lift_for_model('xgb', 'Xgboost')
    _plot_lift_for_model('resn', 'ResNet')

    if all(f'pred_{k}' in plot_train.columns for k in ['xgb', 'resn']) and datasets:
        style = PlotStyle()
        fig, axes = plt.subplots(1, len(datasets), figsize=(11, 5))
        if len(datasets) == 1:
            axes = [axes]
        for ax, (title, data) in zip(axes, datasets):
            plot_double_lift_curve(
                data['pred_xgb'].values,
                data['pred_resn'].values,
                data['w_act'].values,
                data[weight_col].values,
                n_bins=n_bins,
                title=f"Double Lift Chart on {title}",
                label1='Xgboost',
                label2='ResNet',
                pred1_weighted=False,
                pred2_weighted=False,
                actual_weighted=True,
                ax=ax,
                show=False,
                style=style,
            )
        plt.subplots_adjust(wspace=0.3)
        filename = f"02_{model_name}_dlift_xgb_vs_resn.png"
        save_path = _resolve_plot_path(f"{model_name}/double_lift", filename)
        finalize_figure(fig, save_path=save_path, show=False, style=style)
    else:
        print('[Double Lift] Missing predictions; skip.')

print(f'Plots saved under: {output_dir}/plot/{model_name}')
